In [1]:
import pandas as pd
import numpy as np

In [2]:
#Bước 1 - Đọc dữ liệu
df = pd.read_csv("payments.csv",header = 0)
df

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1
...,...,...,...,...
646940,834372,credit_card,35791.36,3
646941,834377,credit_card,36755.53,3
646942,834387,credit_card,59112.48,1
646943,834392,paypal,23836.65,1


In [3]:
#Bước 2 - Chuẩn hóa tên cột & text
# Đưa tên cột về chữ thường, loại bỏ khoảng trắng thừa
df.columns = df.columns.str.strip().str.lower()

# Cắt khoảng trắng và đưa về chữ thường cho cột categorical
df['payment_method'] = df['payment_method'].astype(str).str.strip().str.lower()

print(df.columns.tolist())
print(df['payment_method'].unique())

['order_id', 'payment_method', 'payment_value', 'installments']
['credit_card' 'cod' 'paypal' 'apple_pay' 'bank_transfer']


In [4]:
#Bước 3 - Kiểm tra null, duplicate, dtype
print("=== NULL COUNTS ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ===")
print("Duplicate order_id (PK):", df['order_id'].duplicated().sum())
print("Duplicate toàn dòng:", df.duplicated().sum())

print("\n=== DTYPE ===")
print(df.dtypes)

=== NULL COUNTS ===
order_id          0
payment_method    0
payment_value     0
installments      0
dtype: int64

=== DUPLICATE ===
Duplicate order_id (PK): 0
Duplicate toàn dòng: 0

=== DTYPE ===
order_id            int64
payment_method     object
payment_value     float64
installments        int64
dtype: object


In [5]:
#Bước 4 - Ép kiểu dữ liệu
df = df.astype({
    'order_id': 'int64',
    'installments': 'int64',
    'payment_method': 'category',
    'payment_value': 'float64'
})
print(df.dtypes)

order_id             int64
payment_method    category
payment_value      float64
installments         int64
dtype: object


In [6]:
#Bước 5 - Domain check payment_method
print("Các giá trị payment_method hiện có:")
print(df['payment_method'].value_counts())

# Danh sách hợp lệ theo nghiệp vụ (điều chỉnh nếu có thêm phương thức mới)
valid_methods = {'credit_card', 'cod', 'paypal', 'apple_pay', 'bank_transfer'}
invalid_method = df[~df['payment_method'].isin(valid_methods)]
print("\nSố dòng payment_method KHÔNG nằm trong domain hợp lệ:", invalid_method.shape[0])
invalid_method

Các giá trị payment_method hiện có:
payment_method
credit_card      356352
paypal            97018
cod               96681
apple_pay         64763
bank_transfer     32131
Name: count, dtype: int64

Số dòng payment_method KHÔNG nằm trong domain hợp lệ: 0


,order_id,payment_method,payment_value,installments


In [8]:
#Bước 6 - Làm tròn, lọc bất thường payment_value
before = df.shape[0]

# Làm tròn 2 chữ số thập phân (đơn vị tiền tệ)
df['payment_value'] = df['payment_value'].round(2)

# Lọc bỏ giao dịch âm hoặc bằng 0 (log lại số dòng bị loại)
invalid_value_mask = df['payment_value'] <= 0
print("Số dòng payment_value <= 0 (sẽ bị loại):", invalid_value_mask.sum())

df = df[~invalid_value_mask]
print(f"Số dòng bị loại: {before - df.shape[0]}")

Số dòng payment_value <= 0 (sẽ bị loại): 0
Số dòng bị loại: 0


In [9]:
#Bước 7 - Làm tròn, lọc bất thường installments
print("Phân bố số kỳ trả góp:")
print(df['installments'].value_counts(dropna=False).sort_index())

# Kiểm tra logic nghiệp vụ: installments phải >= 1
invalid_installments = df[df['installments'] < 1]
print("\nSố dòng installments < 1 (bất thường):", invalid_installments.shape[0])
invalid_installments

Phân bố số kỳ trả góp:
installments
1     262866
2       1094
3     218949
6     109910
12     54126
Name: count, dtype: int64

Số dòng installments < 1 (bất thường): 0


,order_id,payment_method,payment_value,installments


In [10]:
#Bước 8 - Xóa duplicate & reset index
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print(f"Loại {before - df.shape[0]} dòng duplicate toàn phần")
print("Shape sau khi làm sạch:", df.shape)

Loại 0 dòng duplicate toàn phần
Shape sau khi làm sạch: (646945, 4)


In [12]:
#Bước 9 - Kiểm tra final & Export
print("Tổng dòng:", df.shape[0])
print("Unique order_id:", df['order_id'].nunique())
print("Null còn sót:\n", df.isnull().sum())
print("payment_value <= 0:", (df['payment_value'] <= 0).sum())
print("installments < 1:", (df['installments'] < 1).sum())
print(df.dtypes)

# Export với utf-8-sig để Excel hiển thị đúng (phòng trường hợp có ký tự đặc biệt sau này)
df.to_csv('silver_payment.csv', index=False, encoding='utf-8-sig')

Tổng dòng: 646945
Unique order_id: 646945
Null còn sót:
 order_id          0
payment_method    0
payment_value     0
installments      0
dtype: int64
payment_value <= 0: 0
installments < 1: 0
order_id             int64
payment_method    category
payment_value      float64
installments         int64
dtype: object
